#Exporting the pandas
created reusable function 
path help to locate the filesw inside the folder 
fild all csv files 
find every files end with csv
combines data frame , pd.concat (stacks the dateframe vertically)
return send the combined dataframe back to the line where the function was called 

In [15]:
import pandas as pd
from pathlib import Path

In [16]:
def read_all_csv(folder_path):
    folder = Path(folder_path)
    csv_files = list(folder.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found in: {folder}"
        )

    frames = []

    for file in csv_files:
        print("Reading:", file.name)
        data = pd.read_csv(file)
        frames.append(data)

    return pd.concat(frames, ignore_index=True)


transactions = read_all_csv(
    r"C:\Users\balaj\OneDrive\Desktop\Transactions_data"
)

locations = read_all_csv(
    r"C:\Users\balaj\OneDrive\Desktop\FPS locations"
)

cards = read_all_csv(
    r"C:\Users\balaj\OneDrive\Desktop\Card status"
)


print("Transactions shape:", transactions.shape)
print("Locations shape:", locations.shape)
print("Cards shape:", cards.shape)

Reading: shop-wise-trans-details_10_2023.csv
Reading: shop-wise-trans-details_10_2024.csv
Reading: shop-wise-trans-details_11_2023.csv
Reading: shop-wise-trans-details_11_2024.csv
Reading: shop-wise-trans-details_12_2023.csv
Reading: shop-wise-trans-details_12_2024.csv
Reading: shop-wise-trans-details_1_2023.csv
Reading: shop-wise-trans-details_1_2024.csv
Reading: shop-wise-trans-details_1_2025.csv
Reading: shop-wise-trans-details_2_2023.csv
Reading: shop-wise-trans-details_2_2024.csv
Reading: shop-wise-trans-details_2_2025.csv
Reading: shop-wise-trans-details_3_2023.csv
Reading: shop-wise-trans-details_3_2024.csv
Reading: shop-wise-trans-details_3_2025.csv
Reading: shop-wise-trans-details_4_2023.csv
Reading: shop-wise-trans-details_4_2024.csv
Reading: shop-wise-trans-details_4_2025.csv
Reading: shop-wise-trans-details_5_2023.csv
Reading: shop-wise-trans-details_5_2024.csv
Reading: shop-wise-trans-details_5_2025.csv
Reading: shop-wise-trans-details_6_2023.csv
Reading: shop-wise-trans-d

In [17]:
# define the data keys 
# we have chosen the shopNo and distcode as shop idetifier and added month as find out the month detection or time identifier 

shop_key =["shopNo","distCode"]
month_key = ["shopNo","distCode","month","year"]

In [ ]:
#standardize the key data types or validatin and cleaning the keys 

for frame in [transactions, locations, cards]: # shope keys have been used for all those and convert the 
    for column in shop_key:
        frame[column]= pd.to_numeric(frame[column],errors="coerce").astype("Int64") #it just converts values to numbers (with NaN for invalid entries)


for frame in [transactions, cards]:
    for column in ["month","year"]:
        frame[column]= pd.to_numeric(frame[column],errors="coerce").astype("Int64") #it just converts values to numbers (with NaN for invalid entries)



print("Transactions missing values:",transactions[month_key].isna().any(axis=1).sum())
print("Locations missing values:",locations[shop_key].isna().any(axis=1).sum())
print("Cards missing values:",cards[month_key].isna().any(axis=1).sum())


#checking real dublicates 

transactions_duplicates = transactions[transactions.duplicated(month_key, keep=False)]
cards_duplicates = cards[cards.duplicated(month_key, keep=False)]
locations_duplicates = locations[locations.duplicated(shop_key, keep=False)]


print("Transactions duplicates:", len(transactions_duplicates))
print("Locations duplicates:", len(locations_duplicates))
print("Cards duplicates:", len(cards_duplicates))
print("Transactions columns:", cards.columns.tolist())
print(transactions.dtypes)

In [19]:
#prepare the monthly features
#these below columns names are mesure based to convert numnerci 
numeric_transactions_columns= [
    "noOfRcs",
    "noOfTrans",
    "riceAfsc",
    "riceFsc",
    "riceAap",
    "wheat",
    "sugar",
    "rgdal",
    "kerosene",
    "totalAmount",
    "salt",
    "otherShopTransCnt"

]

for column in numeric_transactions_columns:
    transactions[column]=pd.to_numeric(transactions[column],errors="coerce")

numeric_card_columns = [
    "totalRcs",
    "totalUnits"
]

for column in numeric_card_columns:
    cards[column] = pd.to_numeric(
        cards[column],
        errors="coerce"
    )



transactions["period"] = pd.to_datetime(
    {
        "year": transactions["year"],
        "month": transactions["month"],
        "day": 1
    },
    errors="coerce"
)

cards["period"] = pd.to_datetime(
    {
        "year": cards["year"],
        "month": cards["month"],
        "day": 1
    },
    errors="coerce"
)

In [20]:
#perform the correct triple join 
card_columns = month_key + [
    "totalRcs",
    "totalUnits",
    "rcNfsaAay",
    "rcNfsaPhh",
    "totalRcNfsa",
    "totalRcState",
    "gasCylinders"
]

card_monthly = cards[
    [column for column in card_columns
     if column in cards.columns]
].copy()


location_columns = shop_key+ [
    "distName",
    "officeCode",
    "officeName",
    "address",
    "longitude",
    "latitude",
    "fpsStatus",
    "fpsType"
]

location_master = locations[
    location_columns
].copy()

In [21]:
#Merge transactions with cards by shop-month:

monthly_unified = transactions.merge(
    card_monthly,
    on=month_key,
    how="outer",
    validate="one_to_one",
    suffixes=("", "_card")
)

#Then attach one location to each shop:
monthly_unified = monthly_unified.merge(
    location_master,
    on=shop_key,
    how="left",
    validate="many_to_one",
    suffixes=("", "_location")
)


In [22]:
monthly_unified["period"] = pd.to_datetime(
    {
        "year": monthly_unified["year"],
        "month": monthly_unified["month"],
        "day": 1
    },
    errors="coerce"
)

print(
    "First period:",
    monthly_unified["period"].min()
)

print(
    "Latest period:",
    monthly_unified["period"].max()
)

First period: 2023-01-01 00:00:00
Latest period: 2025-06-01 00:00:00


In [23]:
monthly_unified["total_rice"] = (
    monthly_unified["riceAfsc"].fillna(0)
    + monthly_unified["riceFsc"].fillna(0)
    + monthly_unified["riceAap"].fillna(0)
)

monthly_unified["utilization_ratio"] = (
    monthly_unified["noOfTrans"]
    / monthly_unified["totalRcs"].replace(0, pd.NA)
)

monthly_unified["portability_ratio"] = (
    monthly_unified["otherShopTransCnt"]
    / monthly_unified["noOfTrans"].replace(0, pd.NA)
)

monthly_unified["rice_wheat_ratio"] = (
    monthly_unified["total_rice"]
    / monthly_unified["wheat"].replace(0, pd.NA)
)

print(
    "Transactions before merge:",
    transactions["noOfTrans"].sum()
)

print(
    "Transactions after merge:",
    monthly_unified["noOfTrans"].sum()
)




Transactions before merge: 222379611
Transactions after merge: 222379611.0


In [24]:

#correct dataset shapes 

print("Transactions;", transactions.shape)
print("Cards:", cards.shape)
print("Locations:", locations.shape)

Transactions; (499648, 20)
Cards: (500264, 29)
Locations: (17434, 11)


In [25]:
#Confirm the correct grains

shop_key = ["distCode", "shopNo"]
month_key = ["distCode", "shopNo", "year", "month"]

print(
    "Location-key duplicates:",
    locations.duplicated(shop_key).sum()
)

print(
    "Transaction monthly-key duplicates:",
    transactions.duplicated(month_key).sum()
)

print(
    "Card monthly-key duplicates:",
    cards.duplicated(month_key).sum()

)

Location-key duplicates: 0
Transaction monthly-key duplicates: 0
Card monthly-key duplicates: 0


In [26]:
#final summary 

notebook_1_summary = pd.Series({
    "unified_monthly_rows": len(monthly_unified),

    "unique_monthly_keys": (
        monthly_unified[
            month_key
        ].drop_duplicates().shape[0]
    ),

    "unique_shops": (
        monthly_unified[
            shop_key
        ].drop_duplicates().shape[0]
    ),

    "first_period": (
        monthly_unified["period"].min()
    ),

    "latest_period": (
        monthly_unified["period"].max()
    ),

    "actual_transactions": (
        monthly_unified["noOfTrans"].sum()
    ),

    "missing_card_months": (
        monthly_unified["totalRcs"].isna().sum()
    ),

    "missing_coordinates": (
        monthly_unified["longitude"].isna()
        | monthly_unified["latitude"].isna()
    ).sum()
})

display(
    notebook_1_summary.to_frame("value")
)

,value
unified_monthly_rows,517651
unique_monthly_keys,517651
unique_shops,17576
first_period,2023-01-01 00:00:00
latest_period,2025-06-01 00:00:00
actual_transactions,222379611.0
missing_card_months,17387
missing_coordinates,5154


In [27]:
MONTHLY_KEY = [
    "distCode",
    "shopNo",
    "year",
    "month"
]

invalid_period_mask = (
    monthly_unified[MONTHLY_KEY]
    .isna()
    .any(axis=1)
    |
    ~monthly_unified["month"].between(
        1,
        12,
        inclusive="both"
    )
    |
    monthly_unified["period"].isna()
)

print(
    "Invalid or missing period rows:",
    invalid_period_mask.sum()
)

Invalid or missing period rows: 0


In [28]:
print(
    monthly_unified.shape
)

(517651, 39)


In [39]:
from pathlib import Path

output_file = Path(
    r"C:\Users\balaj\OneDrive\Desktop\TELANGANA PDS ANALYTICS\data\processed\monthly_unified.csv"
)

monthly_unified .to_csv(output_file, index=False)

